In [ ]:
import os
from pathlib import Path

REVIEW_ROOT = next(
    (path for path in (Path.cwd(), *Path.cwd().parents)
     if (path / "REVIEW_CODE_ROOT").is_file()),
    None,
)
if REVIEW_ROOT is None:
    raise RuntimeError("Open this notebook from the extracted review package.")
os.chdir(REVIEW_ROOT)
REVIEW_FREQUENCY = os.environ.get("BFORTPROB_FREQUENCY", "daily")
if REVIEW_FREQUENCY not in {"daily", "weekly"}:
    raise ValueError("BFORTPROB_FREQUENCY must be daily or weekly.")


# Normalization ablation extended with Student-t GARCH and Timestral

### Run Timestral analysis

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

# ============================================================
# CONFIG
# ============================================================

PROJECT_ROOT = Path(
    r"."
)

NONSTD_ROOT = (
    PROJECT_ROOT
    / "BForTFin_results"
    / "output_daily"
)

STD_ROOT = (
    PROJECT_ROOT
    / "BForTFin_results"
    / "standardised"
    / "output_daily"
)

DATA_ROOT = (
    PROJECT_ROOT
    / "BForTFin_log_return"
    / "arima_split_datasets"
    / "output_daily"
)

OUT_DIR = (
    STD_ROOT
    / "normalisation_sensitivity"
)

OUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

LOOKBACK = 250
HORIZON = 20

QUANTILE_LEVELS = np.arange(1, 10) / 10

ASSET_FOLDERS = {
    "G7": "FX",
    "EM": "EM FX",
    "MSCI": "MSCI",
    "Rates": "Rates",
    "REIT": "REIT",
    "Equities": "Equities",
    "ETF": "ETF",
    "Crypto": "Crypto",
}

MODEL_INFO = {
    "ARIMA": {
        "folder": "arima",
        "suffix": "arima",
        "test_only": True,
        "type": "point",
    },

    "Moirai": {
        "folder": "moirai",
        "suffix": "moirai-2.0-r-small",
        "test_only": True,
        "type": "point",
    },

    "Chronos-T5": {
        "folder": "chronos",
        "suffix": "chronos-t5-small",
        "test_only": False,
        "type": "samples",
    },

    "Lag-Llama": {
        "folder": "lag-llama",
        "suffix": "lag-llama",
        "test_only": False,
        "type": "quantiles",
    },

    "TimesFM": {
        "folder": "timesfm",
        "suffix": "timesfm-2.5-200m",
        "test_only": False,
        "type": "quantiles",
    },

    "Timestral": {
        "folder": "timestral",
        "suffix": "timestral-209m_unstandardized",
        "test_only": True,
        "type": "quantiles",
    },

    "Toto": {
        "folder": "toto",
        "suffix": "toto",
        "test_only": False,
        "type": "quantiles",
    },

    "Sundial": {
        "folder": "sundial",
        "suffix": "sundial",
        "test_only": False,
        "type": "quantiles",
    },
}

SUPERVISED_SUFFIX = (
    "_no_preprocess_lookback-250"
    "_horizon-20_slide-1"
)


# ============================================================
# METRIC HELPERS
# ============================================================

def pinball_loss(y, q, tau):

    error = y - q

    return np.maximum(
        tau * error,
        (tau - 1.0) * error,
    )


def calculate_rmse(y_true, y_pred):

    if y_true.shape != y_pred.shape:
        raise ValueError(
            f"RMSE shape mismatch: "
            f"{y_true.shape} vs {y_pred.shape}"
        )

    rmse_by_horizon = np.sqrt(
        np.mean(
            (y_true - y_pred) ** 2,
            axis=0,
        )
    )

    return float(
        np.mean(rmse_by_horizon)
    )


def calculate_qcrps(y_true, quantiles):

    expected_shape = (
        y_true.shape[0],
        HORIZON,
        len(QUANTILE_LEVELS),
    )

    if quantiles.shape != expected_shape:
        raise ValueError(
            f"Quantile shape mismatch: "
            f"{quantiles.shape} != {expected_shape}"
        )

    errors = []

    for j, tau in enumerate(QUANTILE_LEVELS):

        loss = pinball_loss(
            y_true,
            quantiles[:, :, j],
            tau,
        )

        errors.append(
            np.mean(loss)
        )

    return float(
        2.0 * np.mean(errors)
    )


def calculate_qss(X, Y, quantiles):

    reference_q10 = np.quantile(
        X,
        0.1,
        axis=1,
    )[:, None]

    reference_loss = np.mean(
        pinball_loss(
            Y,
            reference_q10,
            0.1,
        )
    )

    q10_idx = np.argmin(
        np.abs(QUANTILE_LEVELS - 0.1)
    )

    model_loss = np.mean(
        pinball_loss(
            Y,
            quantiles[:, :, q10_idx],
            0.1,
        )
    )

    if reference_loss <= 0:
        raise ValueError(
            "Invalid QSS reference loss."
        )

    return float(
        1.0 - model_loss / reference_loss
    )


# ============================================================
# DATA HELPERS
# ============================================================

def get_dataset_name(test_path):

    name = test_path.stem

    if not name.startswith("test_"):
        raise ValueError(test_path.name)

    name = name.removeprefix("test_")

    if name.endswith(SUPERVISED_SUFFIX):
        name = name[:-len(SUPERVISED_SUFFIX)]

    return name


def load_test_data(test_path):

    df = pd.read_csv(test_path)

    x_cols = [
        f"X_{i}"
        for i in range(LOOKBACK)
    ]

    y_cols = [
        f"Y_{i}"
        for i in range(HORIZON)
    ]

    X = df[x_cols].to_numpy(dtype=float)
    Y = df[y_cols].to_numpy(dtype=float)

    return X, Y


def load_point_csv(path):

    df = pd.read_csv(path)

    y_cols = [
        f"Y_{i}"
        for i in range(HORIZON)
    ]

    missing = [
        c for c in y_cols
        if c not in df.columns
    ]

    if missing:
        raise ValueError(
            f"{path.name}: missing {missing}"
        )

    return df[y_cols].to_numpy(dtype=float)


def load_quantile_npz(path, model_name):

    with np.load(path) as f:

        if "quantiles" not in f.files:
            raise KeyError(
                f"{path.name}: no 'quantiles' key. "
                f"Available keys: {f.files}"
            )

        raw = np.asarray(
            f["quantiles"],
            dtype=float,
        )

        # ============================================
        # TIMESFM
        # Original: (N, 20, 10)
        # First channel is excluded
        # ============================================

        if model_name == "TimesFM":

            if raw.ndim != 3:
                raise ValueError(
                    f"TimesFM: expected 3D array, "
                    f"got {raw.shape}"
                )

            if raw.shape[1:] != (HORIZON, 10):
                raise ValueError(
                    f"TimesFM: expected (N, 20, 10), "
                    f"got {raw.shape}"
                )

            Q = raw[:, :, 1:]

        # ============================================
        # TOTO
        # Original: (9, N, 1, 20)
        # ============================================

        elif model_name == "Toto":

            if raw.ndim != 4:
                raise ValueError(
                    f"Toto: expected 4D array, "
                    f"got {raw.shape}"
                )

            if (
                raw.shape[0] != 9
                or raw.shape[2] != 1
                or raw.shape[3] != HORIZON
            ):
                raise ValueError(
                    f"Toto: expected (9, N, 1, 20), "
                    f"got {raw.shape}"
                )

            Q = raw[:, :, 0, :].transpose(1, 2, 0)

        # ============================================
        # Timestral
        # Original: (N, 9, 20)
        # ============================================

        elif model_name == "Timestral":

            if raw.ndim != 3:
                raise ValueError(
                    f"Timestral: expected 3D array, "
                    f"got {raw.shape}"
                )

            if (
                raw.shape[1] != 9
                or raw.shape[2] != HORIZON
            ):
                raise ValueError(
                    f"Timestral: expected (N, 9, 20), "
                    f"got {raw.shape}"
                )

            Q = raw.transpose(0, 2, 1)

        # ============================================
        # OTHER MODELS
        # Original: (N, 20, 9)
        # ============================================

        else:

            if "quantile_levels" not in f.files:
                raise KeyError(
                    f"{path.name}: "
                    "'quantile_levels' missing. "
                    f"Available keys: {f.files}"
                )

            levels = np.asarray(
                f["quantile_levels"],
                dtype=float,
            )

            if raw.ndim != 3 or raw.shape[1] != HORIZON:
                raise ValueError(
                    f"{model_name}: unexpected shape "
                    f"{raw.shape}"
                )

            indices = []

            for tau in QUANTILE_LEVELS:

                matches = np.flatnonzero(
                    np.isclose(levels, tau)
                )

                if len(matches) != 1:
                    raise ValueError(
                        f"{model_name}: missing or duplicate "
                        f"quantile level {tau}"
                    )

                indices.append(matches[0])

            Q = raw[:, :, indices]

    # ================================================
    # FINAL VALIDATION
    # ================================================

    if Q.ndim != 3 or Q.shape[1:] != (HORIZON, 9):
        raise ValueError(
            f"{model_name}: final quantile shape "
            f"{Q.shape}, expected (N, 20, 9)"
        )

    if not np.isfinite(Q).all():
        raise ValueError(
            f"{model_name}: non-finite quantile values"
        )

    return Q


def select_test_origins(
    predictions,
    n_test,
    already_test_only=False,
):

    if already_test_only:

        if predictions.shape[0] != n_test:
            raise ValueError(
                "Test-only forecast length mismatch: "
                f"{predictions.shape[0]} != {n_test}"
            )

        return predictions

    if predictions.shape[0] < n_test:
        raise ValueError(
            "Not enough forecast origins: "
            f"{predictions.shape[0]} < {n_test}"
        )

    return predictions[-n_test:]


# ============================================================
# EXACT FILE LOOKUP
# ============================================================

def get_forecast_paths(
    model_name,
    folder_name,
    dataset_name,
):

    info = MODEL_INFO[model_name]

    model_dir = (
        NONSTD_ROOT
        / info["folder"]
        / folder_name
    )

    suffix = info["suffix"]

    if info["test_only"]:

        prefix = (
            f"test_{dataset_name}"
            f"{SUPERVISED_SUFFIX}"
            f"_{suffix}"
        )

        if model_name == "ARIMA":
            point_name = f"{prefix}.csv"
        else:
            point_name = f"{prefix}_zero-shot.csv"

        # return {
        #     "point": model_dir / point_name,
        #     "quantiles": None,
        #     "samples": None,
        # }

        return {
            "point": model_dir / point_name,
            "quantiles": model_dir / f"{prefix}_zero-shot_quantiles.npz",
            "samples": None,
        }
        

    prefix = (
        f"{dataset_name}_{suffix}_zero-shot"
    )

    return {
        "point": model_dir / f"{prefix}.csv",
        "quantiles": model_dir / f"{prefix}_quantiles.npz",
        "samples": model_dir / f"{prefix}_samples.npz",
    }

In [ ]:
# ============================================================
# EVALUATE NON-STANDARDISED FORECASTS
# ============================================================

rows = []
errors = []

for model_name, info in MODEL_INFO.items():

    if model_name != "Timestral":
        continue

    print("\n" + "=" * 70)
    print("MODEL:", model_name)
    print("=" * 70)

    for folder_name, asset_class in ASSET_FOLDERS.items():

        test_dir = (
            DATA_ROOT
            / folder_name
        )

        test_files = sorted(
            test_dir.glob("test_*.csv")
        )

        for test_path in test_files:

            dataset = get_dataset_name(test_path)

            if "tether" in dataset.lower():
                continue

            try:

                X, Y = load_test_data(test_path)

                n_test = len(Y)

                paths = get_forecast_paths(
                    model_name,
                    folder_name,
                    dataset,
                )

                point = None
                Q = None

                # ----------------------------------------
                # CHRONOS: reconstruct from samples
                # ----------------------------------------

                if info["type"] == "samples":

                    sample_path = paths["samples"]

                    if not sample_path.exists():
                        raise FileNotFoundError(
                            sample_path
                        )

                    with np.load(sample_path) as f:

                        samples = np.asarray(
                            f["samples"],
                            dtype=float,
                        )

                    if samples.ndim != 3:
                        raise ValueError(
                            f"Unexpected samples shape: "
                            f"{samples.shape}"
                        )

                    if samples.shape[2] != HORIZON:
                        raise ValueError(
                            f"Wrong sample horizon: "
                            f"{samples.shape}"
                        )

                    samples = select_test_origins(
                        samples,
                        n_test,
                    )

                    point = np.median(
                        samples,
                        axis=1,
                    )

                    Q = np.quantile(
                        samples,
                        QUANTILE_LEVELS,
                        axis=1,
                    )

                    Q = np.moveaxis(
                        Q,
                        0,
                        -1,
                    )

                # ----------------------------------------
                # OTHER MODELS
                # ----------------------------------------

                else:

                    point_path = paths["point"]

                    if not point_path.exists():
                        raise FileNotFoundError(
                            point_path
                        )

                    point = load_point_csv(
                        point_path
                    )

                    point = select_test_origins(
                        point,
                        n_test,
                        already_test_only=info["test_only"],
                    )

                    if info["type"] == "quantiles":

                        quantile_path = paths["quantiles"]

                        if not quantile_path.exists():
                            raise FileNotFoundError(
                                quantile_path
                            )

                        Q = load_quantile_npz(
                            quantile_path,
                            model_name,
                        )

                        Q = select_test_origins(
                            Q,
                            n_test,
                        )

                # ----------------------------------------
                # VALIDATE
                # ----------------------------------------

                if point.shape != Y.shape:
                    raise ValueError(
                        f"Point shape mismatch: "
                        f"{point.shape} vs {Y.shape}"
                    )

                if not np.isfinite(point).all():
                    raise ValueError(
                        "Non-finite point predictions."
                    )

                if Q is not None:

                    if Q.shape != (
                        n_test,
                        HORIZON,
                        9,
                    ):
                        raise ValueError(
                            f"Quantile shape mismatch: {Q.shape}"
                        )

                    if not np.isfinite(Q).all():
                        raise ValueError(
                            "Non-finite quantiles."
                        )

                    if np.any(
                        np.diff(Q, axis=2) < -1e-10
                    ):
                        raise ValueError(
                            "Quantile crossing detected."
                        )

                # ----------------------------------------
                # METRICS
                # ----------------------------------------

                rmse = calculate_rmse(
                    Y,
                    point,
                )

                qcrps = np.nan
                qss = np.nan

                if Q is not None:

                    qcrps = calculate_qcrps(
                        Y,
                        Q,
                    )

                    qss = calculate_qss(
                        X,
                        Y,
                        Q,
                    )

                rows.append({
                    "model": model_name,
                    "asset_class": asset_class,
                    "folder": folder_name,
                    "dataset": dataset,
                    "n_test_origins": n_test,
                    "rmse_no_scaler": rmse,
                    "qcrps_no_scaler": qcrps,
                    "qss_no_scaler": qss,
                })

                print(
                    f"  {dataset}: "
                    f"RMSE={rmse:.6f}, "
                    f"qCRPS={qcrps:.6f}, "
                    f"QSS={qss:.6f}"
                )

            except Exception as exc:

                errors.append({
                    "model": model_name,
                    "asset_class": asset_class,
                    "dataset": dataset,
                    "error": str(exc),
                })

                print(
                    f"  ERROR {dataset}: {exc}"
                )


# ============================================================
# SAVE
# ============================================================

nonstd_results = pd.DataFrame(rows)
error_df = pd.DataFrame(errors)

nonstd_results.to_csv(
    OUT_DIR
    / "daily_nonstandardised_metrics_by_dataset_timestral_only.csv",
    index=False,
)

error_df.to_csv(
    OUT_DIR
    / "daily_nonstandardised_errors_timestral_only.csv",
    index=False,
)

print("\n" + "=" * 80)
print("NON-STANDARDISED EVALUATION SUMMARY")
print("=" * 80)

if len(nonstd_results):

    print(
        nonstd_results.groupby("model").agg(
            n_datasets=("dataset", "count"),
            mean_rmse=("rmse_no_scaler", "mean"),
            mean_qcrps=("qcrps_no_scaler", "mean"),
            mean_qss=("qss_no_scaler", "mean"),
        ).to_string()
    )

print("\nErrors:", len(error_df))

if len(error_df):
    print(error_df.to_string(index=False))

### Combine with old results

These are still excluding Student-t GARCH.

In [ ]:
# ============================================================
# COMBINE OLD + NEW RESULTS
# ============================================================

old = pd.read_csv(
    OUT_DIR / "daily_no_scaler_all_11_models.csv"
)

new = pd.read_csv(
    OUT_DIR / "daily_nonstandardised_metrics_by_dataset_timestral_only.csv"
)

# Combine
combined = pd.concat(
    [old, new],
    ignore_index=True,
)

keys = ["model", "asset_class", "dataset"]

if combined.duplicated(keys).any():
    raise ValueError(
        "Duplicate model–dataset combinations."
    )

counts = combined.groupby("model")[
    "dataset"
].count()

print(counts.to_string())

# Require complete coverage before treating this
# as the final daily ablation.
expected_models = [
    "ARIMA",
    "Chronos-T5",
    "Lag-Llama",
    "Sundial",
    "TimesFM",
    "Timestral",
    "Toto",
    "LightGBM",
    "GARCH(1,1)",
    "LSTM",
    "Moirai",
    "Moirai-MoE",
]

if set(counts.index) != set(expected_models):
    raise ValueError(
        "Missing or unexpected models."
    )

if not (counts == 72).all():
    raise ValueError(
        "Some models do not have all 72 datasets."
    )

combined.to_csv(
    OUT_DIR / "daily_no_scaler_all_12_models.csv",
    index=False,
)

print("\nTotal evaluations:", len(combined))

# Extend results

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

# ============================================================
# CONFIGURATION
# ============================================================

PROJECT_ROOT = Path(
    r"."
)

DATA_ROOT = (
    PROJECT_ROOT
    / "BForTFin_log_return"
    / "arima_split_datasets"
    / "output_daily"
)

ABLATION_ROOT = (
    PROJECT_ROOT
    / "BForTFin_results"
    / "normalisation_ablation_no_scaler"
    / "output_daily"
)

OUT_DIR = (
    PROJECT_ROOT
    / "BForTFin_results"
    / "standardised"
    / "output_daily"
    / "normalisation_sensitivity"
)

OUT_DIR.mkdir(parents=True, exist_ok=True)

ASSET_FOLDERS = {
    "G7": "FX",
    "EM": "EM FX",
    "MSCI": "MSCI",
    "Rates": "Rates",
    "REIT": "REIT",
    "Equities": "Equities",
    "ETF": "ETF",
    "Crypto": "Crypto",
}

MODELS = [
    "LightGBM",
    "GARCH(1,1)",
    "LSTM",
    "Moirai",
    "Moirai-MoE",
    "Student-t GARCH",
]

HORIZON = 20
LOOKBACK = 250
LEVELS = np.arange(1, 10) / 10

SUPERVISED_SUFFIX = (
    "_no_preprocess_lookback-250"
    "_horizon-20_slide-1"
)


# ============================================================
# METRICS
# ============================================================

def pinball(y, q, tau):
    e = y - q
    return np.maximum(tau * e, (tau - 1) * e)


def rmse_metric(Y, P):
    if Y.shape != P.shape:
        raise ValueError(
            f"RMSE shape mismatch: {Y.shape} vs {P.shape}"
        )

    # Match original horizon-wise RMSE aggregation
    return float(
        np.sqrt(np.mean((Y - P) ** 2, axis=0)).mean()
    )


def qcrps_metric(Y, Q):
    if Q.shape != (len(Y), HORIZON, 9):
        raise ValueError(
            f"Unexpected quantile shape: {Q.shape}"
        )

    losses = [
        pinball(Y, Q[:, :, j], tau).mean()
        for j, tau in enumerate(LEVELS)
    ]

    return float(2 * np.mean(losses))


def qss_metric(X, Y, Q):
    ref_q10 = np.quantile(X, 0.1, axis=1)[:, None]

    ref_loss = pinball(Y, ref_q10, 0.1).mean()
    model_loss = pinball(Y, Q[:, :, 0], 0.1).mean()

    if ref_loss <= 0:
        raise ValueError("Invalid reference pinball loss")

    return float(1 - model_loss / ref_loss)


# ============================================================
# DATASET HELPERS
# ============================================================

def dataset_from_test_path(path):
    name = path.stem

    if not name.startswith("test_"):
        raise ValueError(path.name)

    name = name[len("test_"):]

    if not name.endswith(SUPERVISED_SUFFIX):
        raise ValueError(path.name)

    return name[:-len(SUPERVISED_SUFFIX)]


def load_targets(path):
    df = pd.read_csv(path)

    xcols = [f"X_{i}" for i in range(LOOKBACK)]
    ycols = [f"Y_{i}" for i in range(HORIZON)]

    X = df[xcols].to_numpy(dtype=float)
    Y = df[ycols].to_numpy(dtype=float)

    return X, Y


def load_points(path):
    if not path.exists():
        raise FileNotFoundError(path)

    df = pd.read_csv(path)
    cols = [f"Y_{i}" for i in range(HORIZON)]

    return df[cols].to_numpy(dtype=float)


def load_quantiles(path, model):
    if not path.exists():
        raise FileNotFoundError(path)

    with np.load(path) as f:
        raw = np.asarray(f["quantiles"], dtype=float)

        if "quantile_levels" not in f.files:
            raise ValueError(
                f"{path.name}: missing quantile_levels"
            )

        levels = np.asarray(
            f["quantile_levels"],
            dtype=float
        )

    # Moirai 2.0 uses (N, Q, H)
    if model == "Moirai":

        if (
            raw.ndim != 3
            or raw.shape[2] != HORIZON
            or raw.shape[1] != len(levels)
        ):
            raise ValueError(
                f"Moirai quantile shape: {raw.shape}"
            )

        raw = raw.transpose(0, 2, 1)

    else:
        # LightGBM and GARCH: (N, H, Q)
        if (
            raw.ndim != 3
            or raw.shape[1] != HORIZON
            or raw.shape[2] != len(levels)
        ):
            raise ValueError(
                f"{model} quantile shape: {raw.shape}"
            )

    indices = []

    for tau in LEVELS:
        matches = np.flatnonzero(
            np.isclose(levels, tau)
        )

        if len(matches) != 1:
            raise ValueError(
                f"{path.name}: missing/duplicate tau={tau}"
            )

        indices.append(int(matches[0]))

    return raw[:, :, indices]


# ============================================================
# MODEL-SPECIFIC PATHS
# ============================================================

def get_paths(model, folder, dataset, test_stem):

    if model == "LightGBM":

        base = (
            ABLATION_ROOT
            / "lightgbm"
            / folder
            / f"{test_stem}_lightgbm"
        )

        return (
            Path(f"{base}.csv"),
            Path(f"{base}_quantiles.npz"),
        )

    if model == "LSTM":

        base = (
            ABLATION_ROOT
            / "lstm"
            / folder
            / f"{test_stem}_lstm.csv"
        )

        return base, None

    if model == "GARCH(1,1)":

        folder_path = (
            ABLATION_ROOT / "garch" / folder
        )

        point_path = (
            folder_path
            / f"{test_stem}_garch-1-1.csv"
        )

        clean_name = (
            dataset.lower()
            .replace(" historical data", "")
            .replace(" ", "_")
            .replace(".", "")
        )

        quantile_path = (
            folder_path
            / (
                f"{folder.lower()}_{clean_name}"
                f"_garch_1_1_train_test_quantiles.npz"
            )
        )

        return point_path, quantile_path

    if model == "Moirai":

        base = (
            ABLATION_ROOT
            / "moirai"
            / folder
            / (
                f"{test_stem}"
                "_moirai-2.0-r-small_no_scaler_zero-shot"
            )
        )

        return (
            Path(f"{base}.csv"),
            Path(f"{base}_quantiles.npz"),
        )

    if model == "Moirai-MoE":

        base = (
            ABLATION_ROOT
            / "moirai-moe"
            / folder
            / (
                f"{test_stem}"
                "_moirai-moe-1.0-r-base_no_scaler_zero-shot"
            )
        )

        return (
            Path(f"{base}.csv"),
            Path(f"{base}_samples.npz"),
        )

    if model == "Student-t GARCH":

        base = (
            ABLATION_ROOT
            / "garch_t_1_1"
            / folder
            / (
                f"{test_stem}"
                "_garch-t-1-1-simulated"
            )
        )

        return (
            Path(f"{base}.csv"),
            Path(f"{base}_quantiles.npz"),
        )

    raise ValueError(model)


# ============================================================
# EVALUATE
# ============================================================

rows = []
errors = []

for model in MODELS:

    if model != "Student-t GARCH":
        continue

    print("\n" + "=" * 70)
    print(model)
    print("=" * 70)

    for folder, asset_class in ASSET_FOLDERS.items():

        test_dir = DATA_ROOT / folder

        test_files = sorted(
            test_dir.glob(
                f"test_*{SUPERVISED_SUFFIX}.csv"
            )
        )

        for test_path in test_files:

            dataset = dataset_from_test_path(test_path)

            try:
                X, Y = load_targets(test_path)

                point_path, probabilistic_path = get_paths(
                    model,
                    folder,
                    dataset,
                    test_path.stem,
                )

                P = load_points(point_path)

                if P.shape != Y.shape:
                    raise ValueError(
                        f"Point: {P.shape} vs {Y.shape}"
                    )

                if not np.isfinite(P).all():
                    raise ValueError(
                        "Non-finite point forecasts"
                    )

                rmse = rmse_metric(Y, P)

                qcrps = np.nan
                qss = np.nan

                if model == "Moirai-MoE":

                    if not probabilistic_path.exists():
                        raise FileNotFoundError(
                            probabilistic_path
                        )

                    with np.load(probabilistic_path) as f:
                        samples = np.asarray(
                            f["samples"],
                            dtype=float
                        )

                    if (
                        samples.ndim != 3
                        or samples.shape[0] != len(Y)
                        or samples.shape[2] != HORIZON
                    ):
                        raise ValueError(
                            f"Sample shape: {samples.shape}"
                        )

                    if not np.isfinite(samples).all():
                        raise ValueError(
                            "Non-finite predictive samples"
                        )

                    Q = np.quantile(
                        samples,
                        LEVELS,
                        axis=1
                    )

                    Q = np.moveaxis(Q, 0, -1)

                elif probabilistic_path is not None:

                    Q = load_quantiles(
                        probabilistic_path,
                        model
                    )

                else:
                    Q = None

                if Q is not None:

                    if Q.shape != (len(Y), HORIZON, 9):
                        raise ValueError(
                            f"Quantiles: {Q.shape}"
                        )

                    if not np.isfinite(Q).all():
                        raise ValueError(
                            "Non-finite quantile forecasts"
                        )

                    qcrps = qcrps_metric(Y, Q)
                    qss = qss_metric(X, Y, Q)

                rows.append({
                    "model": model,
                    "asset_class": asset_class,
                    "dataset": dataset,
                    "n_test_origins": len(Y),
                    "rmse_no_scaler": rmse,
                    "qcrps_no_scaler": qcrps,
                    "qss_no_scaler": qss,
                })

            except Exception as exc:

                errors.append({
                    "model": model,
                    "asset_class": asset_class,
                    "dataset": dataset,
                    "error": str(exc),
                })

    successful = sum(
        row["model"] == model for row in rows
    )

    failed = sum(
        row["model"] == model for row in errors
    )

    print(
        f"Successful: {successful} | Failed: {failed}"
    )


# ============================================================
# SAVE RESULTS
# ============================================================

new_results = pd.DataFrame(rows)
error_df = pd.DataFrame(errors)

new_results.to_csv(
    OUT_DIR / "daily_new_no_scaler_metrics_student_t_only.csv",
    index=False
)

error_df.to_csv(
    OUT_DIR / "daily_new_no_scaler_errors_student_t_only.csv",
    index=False
)

print("\nNEW RESULTS")
print(
    new_results.groupby("model").agg(
        n_datasets=("dataset", "count"),
        mean_rmse=("rmse_no_scaler", "mean"),
        mean_qcrps=("qcrps_no_scaler", "mean"),
        mean_qss=("qss_no_scaler", "mean"),
    ).to_string()
)

print("\nERRORS:", len(error_df))

if len(error_df):
    print(error_df.to_string(index=False))

In [ ]:
# ============================================================
# COMBINE OLD + NEW RESULTS
# ============================================================

old = pd.read_csv(
    OUT_DIR / "daily_no_scaler_all_12_models.csv"
)

new = pd.read_csv(
    OUT_DIR / "daily_new_no_scaler_metrics_student_t_only.csv"
)


# Combine
combined = pd.concat(
    [old, new],
    ignore_index=True,
)

keys = ["model", "asset_class", "dataset"]

if combined.duplicated(keys).any():
    raise ValueError(
        "Duplicate model–dataset combinations."
    )

counts = combined.groupby("model")[
    "dataset"
].count()

print(counts.to_string())

# Require complete coverage before treating this
# as the final daily ablation.
expected_models = [
    "ARIMA",
    "Chronos-T5",
    "Lag-Llama",
    "Sundial",
    "TimesFM",
    "Timestral",
    "Toto",
    "LightGBM",
    "GARCH(1,1)",
    "Student-t GARCH",
    "LSTM",
    "Moirai",
    "Moirai-MoE",
]

if set(counts.index) != set(expected_models):
    raise ValueError(
        "Missing or unexpected models."
    )

if not (counts == 72).all():
    raise ValueError(
        "Some models do not have all 72 datasets."
    )

combined.to_csv(
    OUT_DIR / "daily_no_scaler_all_13_models.csv",
    index=False,
)

print("\nTotal evaluations:", len(combined))

# Updated paired comparisons

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

# ============================================================
# CONFIG
# ============================================================

PROJECT_ROOT = Path(
    r"."
)

STD_ROOT = (
    PROJECT_ROOT
    / "BForTFin_results"
    / "standardised"
    / "output_daily"
)

OUT_DIR = (
    STD_ROOT
    / "normalisation_sensitivity"
)

OUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

MODELS = [
    "ARIMA",
    "Chronos-T5",
    "Lag-Llama",
    "Sundial",
    "TimesFM",
    "Timestral",
    "Toto",
    "LightGBM",
    "GARCH(1,1)",
    "Student-t GARCH",
    "LSTM",
    "Moirai",
    "Moirai-MoE",
]

POINT_MODELS = [
    "ARIMA",
    "Chronos-T5",
    "Lag-Llama",
    "Sundial",
    "TimesFM",
    "Toto",
    "LightGBM",
    "Timestral",
    "LSTM",
    "Moirai",
    "Moirai-MoE",
]

PROB_MODELS = [
    "Chronos-T5",
    "Lag-Llama",
    "Sundial",
    "TimesFM",
    "Timestral",
    "Toto",
    "LightGBM",
    "GARCH(1,1)",
    "Student-t GARCH",
    "Moirai",
    "Moirai-MoE",
]

EXPECTED_DATASETS = 72

KEYS = ["model", "asset_class", "dataset"]

SUPERVISED_SUFFIX = (
    "_no_preprocess_lookback-250"
    "_horizon-20_slide-1"
)


# ============================================================
# NORMALISE DATASET NAMES
# ============================================================

def clean_dataset_name(name):

    name = str(name).strip()

    if name.startswith("test_"):
        name = name.removeprefix("test_")

    if name.endswith(SUPERVISED_SUFFIX):
        name = name[:-len(SUPERVISED_SUFFIX)]

    return name


def prepare_keys(df):

    df = df.copy()

    df["dataset"] = (
        df["dataset"]
        .apply(clean_dataset_name)
    )

    df["model"] = (
        df["model"]
        .astype(str)
        .str.strip()
    )

    df["asset_class"] = (
        df["asset_class"]
        .astype(str)
        .str.strip()
    )

    return df


# ============================================================
# LOAD NON-STANDARDISED METRICS
# ============================================================

nonstd = pd.read_csv(
    OUT_DIR / "daily_no_scaler_all_13_models.csv"
)

nonstd = prepare_keys(nonstd)

nonstd = nonstd[
    nonstd["model"].isin(MODELS)
].copy()


# ============================================================
# LOAD STANDARDISED RMSE
# ============================================================

std_rmse = pd.read_csv(
    STD_ROOT
    / "rmse_summary_all_models_with_timestral.csv"
)

std_rmse = prepare_keys(std_rmse)

std_rmse = (
    std_rmse[
        std_rmse["model"].isin(POINT_MODELS)
    ]
    .rename(
        columns={
            "rmse": "rmse_scaler"
        }
    )
)


# ============================================================
# LOAD STANDARDISED qCRPS
# ============================================================

std_crps = pd.read_csv(
    STD_ROOT
    / "all_models_crps_by_dataset_with_timestral.csv"
)

std_crps = prepare_keys(std_crps)

std_crps = (
    std_crps[
        std_crps["model"].isin(PROB_MODELS)
    ]
    .rename(
        columns={
            "mean_qcrps": "qcrps_scaler"
        }
    )
)


# ============================================================
# LOAD STANDARDISED QSS
# ============================================================

std_qss = pd.read_csv(
    STD_ROOT
    / "qss_summary_all_models_with_timestral.csv"
)

std_qss = prepare_keys(std_qss)

std_qss = (
    std_qss[
        std_qss["model"].isin(PROB_MODELS)
    ]
    .rename(
        columns={
            "qss_0.1": "qss_scaler"
        }
    )
)


# ============================================================
# CHECK UNIQUENESS
# ============================================================

for name, df in [
    ("Non-standardised", nonstd),
    ("Standardised RMSE", std_rmse),
    ("Standardised qCRPS", std_crps),
    ("Standardised QSS", std_qss),
]:

    duplicates = df.duplicated(
        subset=KEYS,
        keep=False,
    )

    if duplicates.any():
        raise ValueError(
            f"Duplicate keys in {name}:\n"
            f"{df.loc[duplicates, KEYS].to_string(index=False)}"
        )


# ============================================================
# CHECK DATASET COVERAGE
# ============================================================

for name, df, model_list in [
    ("Non-standardised", nonstd, MODELS),
    ("Standardised RMSE", std_rmse, POINT_MODELS),
    ("Standardised qCRPS", std_crps, PROB_MODELS),
    ("Standardised QSS", std_qss, PROB_MODELS),
]:

    counts = (
        df.groupby("model")["dataset"]
        .count()
    )

    print(f"\n{name}:")
    print(counts.to_string())

    for model in model_list:

        n = counts.get(model, 0)

        if n != EXPECTED_DATASETS:
            raise ValueError(
                f"{name}, {model}: "
                f"expected {EXPECTED_DATASETS}, got {n}"
            )


# ============================================================
# MERGE STANDARDISED METRICS
# Preserve models missing RMSE, such as GARCH
# ============================================================

std = (
    std_rmse[
        KEYS + ["rmse_scaler"]
    ]
    .merge(
        std_crps[
            KEYS + ["qcrps_scaler"]
        ],
        on=KEYS,
        how="outer",
        validate="one_to_one",
    )
    .merge(
        std_qss[
            KEYS + ["qss_scaler"]
        ],
        on=KEYS,
        how="outer",
        validate="one_to_one",
    )
)


# ============================================================
# PAIRED MERGE
# ============================================================

comparison = nonstd[
    KEYS + [
        "n_test_origins",
        "rmse_no_scaler",
        "qcrps_no_scaler",
        "qss_no_scaler",
    ]
].merge(
    std,
    on=KEYS,
    how="outer",
    validate="one_to_one",
    indicator=True,
)

if not (
    comparison["_merge"] == "both"
).all():

    missing = comparison[
        comparison["_merge"] != "both"
    ]

    raise ValueError(
        "Missing paired model–dataset combinations:\n"
        + missing.to_string(index=False)
    )

comparison = comparison.drop(
    columns="_merge"
)

# Ensure probabilistic columns exist for all probabilistic models
prob_rows = comparison[
    comparison["model"].isin(PROB_MODELS)
]

for col in [
    "qcrps_no_scaler",
    "qcrps_scaler",
    "qss_no_scaler",
    "qss_scaler",
]:

    if prob_rows[col].isna().any():
        raise ValueError(
            f"Missing {col} in probabilistic model rows."
        )


# ============================================================
# PAIRED DIFFERENCES
# Positive = STANDARDISATION BETTER
# ============================================================

comparison["delta_rmse"] = (
    comparison["rmse_no_scaler"]
    - comparison["rmse_scaler"]
)

comparison["delta_qcrps"] = (
    comparison["qcrps_no_scaler"]
    - comparison["qcrps_scaler"]
)

comparison["delta_qss"] = (
    comparison["qss_scaler"]
    - comparison["qss_no_scaler"]
)


# ============================================================
# RELATIVE DIFFERENCES (%)
# ============================================================

comparison["relative_rmse_pct"] = (
    100.0
    * comparison["delta_rmse"]
    / comparison["rmse_no_scaler"]
)

comparison["relative_qcrps_pct"] = (
    100.0
    * comparison["delta_qcrps"]
    / comparison["qcrps_no_scaler"]
)


# ============================================================
# STANDARDISATION WINS
# ============================================================

comparison["rmse_scaler_better"] = (
    comparison["delta_rmse"] > 0
)

comparison["qcrps_scaler_better"] = (
    comparison["delta_qcrps"] > 0
)

comparison["qss_scaler_better"] = (
    comparison["delta_qss"] > 0
)


# ============================================================
# MODEL-LEVEL SUMMARY
# ============================================================

summary = (
    comparison
    .groupby("model", as_index=False)
    .agg(
        n_datasets=("dataset", "count"),

        mean_rmse_scaler=("rmse_scaler", "mean"),
        mean_rmse_no_scaler=("rmse_no_scaler", "mean"),

        median_relative_rmse_pct=(
            "relative_rmse_pct", "median"
        ),

        pct_rmse_scaler_better=(
            "rmse_scaler_better", "mean"
        ),

        mean_qcrps_scaler=("qcrps_scaler", "mean"),
        mean_qcrps_no_scaler=("qcrps_no_scaler", "mean"),

        median_relative_qcrps_pct=(
            "relative_qcrps_pct", "median"
        ),

        pct_qcrps_scaler_better=(
            "qcrps_scaler_better", "mean"
        ),

        mean_qss_scaler=("qss_scaler", "mean"),
        mean_qss_no_scaler=("qss_no_scaler", "mean"),

        mean_delta_qss=("delta_qss", "mean"),

        pct_qss_scaler_better=(
            "qss_scaler_better", "mean"
        ),
    )
)

for col in [
    "pct_rmse_scaler_better",
    "pct_qcrps_scaler_better",
    "pct_qss_scaler_better",
]:

    summary[col] *= 100.0


# ============================================================
# SAVE
# ============================================================

comparison.to_csv(
    OUT_DIR
    / "daily_normalisation_paired_by_dataset_all_models_with_timestral.csv",
    index=False,
)

summary.to_csv(
    OUT_DIR
    / "daily_normalisation_model_summary_all_models_with_timestral.csv",
    index=False,
)


# ============================================================
# PRINT
# ============================================================

print("\n" + "=" * 100)
print("DAILY NORMALISATION SENSITIVITY")
print("=" * 100)

print(
    summary.round(6).to_string(index=False)
)

print("\nTotal paired comparisons:", len(comparison))

### Investigate GARCH

In [ ]:
# ============================================================
# GARCH NORMALISATION DIAGNOSTICS
# ============================================================

garch = comparison[
    comparison["model"] == "GARCH(1,1)"
].copy()

garch["qcrps_ratio"] = (
    garch["qcrps_no_scaler"]
    / garch["qcrps_scaler"]
)

garch["abs_qss_difference"] = np.abs(
    garch["qss_no_scaler"]
    - garch["qss_scaler"]
)

print("=" * 80)
print("GARCH qCRPS RATIO")
print("=" * 80)

print(
    garch["qcrps_ratio"].describe(
        percentiles=[0.5, 0.9, 0.95, 0.99]
    )
)

print("\n" + "=" * 80)
print("GARCH QSS DIFFERENCES")
print("=" * 80)

print(
    garch["abs_qss_difference"].describe(
        percentiles=[0.5, 0.9, 0.95, 0.99]
    )
)

print("\n" + "=" * 80)
print("WORST 15 GARCH DATASETS")
print("=" * 80)

cols = [
    "asset_class",
    "dataset",
    "qcrps_scaler",
    "qcrps_no_scaler",
    "qcrps_ratio",
    "qss_scaler",
    "qss_no_scaler",
]

print(
    garch.sort_values(
        "qcrps_ratio",
        ascending=False
    )[cols].head(15).to_string(index=False)
)

print("\n" + "=" * 80)
print("NUMBER OF LARGE DEGRADATIONS")
print("=" * 80)

for threshold in [2, 5, 10]:

    count = (
        garch["qcrps_ratio"] > threshold
    ).sum()

    print(
        f"qCRPS > {threshold}x standardised: "
        f"{count}/72"
    )

In [ ]:
# ============================================================
# GARCH EXTREME-FAILURE DIAGNOSTICS
# ============================================================

import numpy as np
import pandas as pd

garch = comparison[
    comparison["model"] == "GARCH(1,1)"
].copy()

garch["qcrps_ratio"] = (
    garch["qcrps_no_scaler"]
    / garch["qcrps_scaler"]
)

garch["qss_difference"] = (
    garch["qss_no_scaler"]
    - garch["qss_scaler"]
)

# Separate the extreme cases for diagnostic reporting.
garch["extreme_failure"] = (
    garch["qcrps_ratio"] > 10
)

print("GARCH sensitivity by failure group:\n")

print(
    garch.groupby("extreme_failure").agg(
        n_datasets=("dataset", "count"),
        median_qcrps_ratio=("qcrps_ratio", "median"),
        mean_qcrps_ratio=("qcrps_ratio", "mean"),
        median_qss_difference=("qss_difference", "median"),
        mean_qss_difference=("qss_difference", "mean"),
    ).to_string()
)

# Descriptive sensitivity among datasets without
# >10x qCRPS deterioration.
remaining = garch[
    ~garch["extreme_failure"]
]

print("\nRemaining datasets:", len(remaining))

print(
    "Median qCRPS ratio:",
    remaining["qcrps_ratio"].median()
)

print(
    "Median absolute QSS difference:",
    np.median(
        np.abs(remaining["qss_difference"])
    )
)

# Save the diagnostic labels, without modifying
# or excluding anything from the original results.
garch.to_csv(
    OUT_DIR / "daily_garch_failure_diagnostics.csv",
    index=False,
)

## Investigate Student-t GARCH

In [ ]:
# ============================================================
# GARCH NORMALISATION DIAGNOSTICS
# ============================================================

garch_t_1_1 = comparison[
    comparison["model"] == "Student-t GARCH"
].copy()

garch_t_1_1["qcrps_ratio"] = (
    garch_t_1_1["qcrps_no_scaler"]
    / garch_t_1_1["qcrps_scaler"]
)

garch_t_1_1["abs_qss_difference"] = np.abs(
    garch_t_1_1["qss_no_scaler"]
    - garch_t_1_1["qss_scaler"]
)

print("=" * 80)
print("STUDENT-T GARCH qCRPS RATIO")
print("=" * 80)

print(
    garch_t_1_1["qcrps_ratio"].describe(
        percentiles=[0.5, 0.9, 0.95, 0.99]
    )
)

print("\n" + "=" * 80)
print("STUDENT-T GARCH QSS DIFFERENCES")
print("=" * 80)

print(
    garch_t_1_1["abs_qss_difference"].describe(
        percentiles=[0.5, 0.9, 0.95, 0.99]
    )
)

print("\n" + "=" * 80)
print("WORST 17 STUDENT-T GARCH DATASETS")
print("=" * 80)

cols = [
    "asset_class",
    "dataset",
    "qcrps_scaler",
    "qcrps_no_scaler",
    "qcrps_ratio",
    "qss_scaler",
    "qss_no_scaler",
]

print(
    garch_t_1_1.sort_values(
        "qcrps_ratio",
        ascending=False
    )[cols].head(17).to_string(index=False)
)

print("\n" + "=" * 80)
print("NUMBER OF LARGE DEGRADATIONS")
print("=" * 80)

for threshold in [2, 5, 10]:

    count = (
        garch_t_1_1["qcrps_ratio"] > threshold
    ).sum()

    print(
        f"qCRPS > {threshold}x standardised: "
        f"{count}/72"
    )

In [ ]:
# ============================================================
# STUDENT-T GARCH EXTREME-FAILURE DIAGNOSTICS
# ============================================================

import numpy as np
import pandas as pd

garch_t_1_1 = comparison[
    comparison["model"] == "Student-t GARCH"
].copy()

garch_t_1_1["qcrps_ratio"] = (
    garch_t_1_1["qcrps_no_scaler"]
    / garch_t_1_1["qcrps_scaler"]
)

garch_t_1_1["qss_difference"] = (
    garch_t_1_1["qss_no_scaler"]
    - garch_t_1_1["qss_scaler"]
)

# Separate the extreme cases for diagnostic reporting.
garch_t_1_1["extreme_failure"] = (
    garch_t_1_1["qcrps_ratio"] > 10
)

print("Student-t GARCH sensitivity by failure group:\n")

print(
    garch_t_1_1.groupby("extreme_failure").agg(
        n_datasets=("dataset", "count"),
        median_qcrps_ratio=("qcrps_ratio", "median"),
        mean_qcrps_ratio=("qcrps_ratio", "mean"),
        median_qss_difference=("qss_difference", "median"),
        mean_qss_difference=("qss_difference", "mean"),
    ).to_string()
)

# Descriptive sensitivity among datasets without
# >10x qCRPS deterioration.
remaining = garch_t_1_1[
    ~garch_t_1_1["extreme_failure"]
]

print("\nRemaining datasets:", len(remaining))

print(
    "Median qCRPS ratio:",
    remaining["qcrps_ratio"].median()
)

print(
    "Median absolute QSS difference:",
    np.median(
        np.abs(remaining["qss_difference"])
    )
)

# Save the diagnostic labels, without modifying
# or excluding anything from the original results.
garch_t_1_1.to_csv(
    OUT_DIR / "daily_garch_t_1_1_failure_diagnostics.csv",
    index=False,
)

### Compare scale-only Student-t GARCH vs StandardScaler Student-t GARCH

In [ ]:
PROJECT_ROOT = Path(r".")
FREQUENCY = "daily"
BASE = PROJECT_ROOT / "BForTFin_results"
SCALE_ROOT = BASE / "normalisation_ablation_scale_only" / f"output_{FREQUENCY}" / "garch_t_1_1"
STD_ROOT = BASE / "standardised" / f"output_{FREQUENCY}" / "garch_t_1_1_standardized_train_test"
DATA_ROOT = PROJECT_ROOT / "BForTFin_log_return" / "arima_split_datasets" / f"output_{FREQUENCY}"
OUT_DIR = BASE / "normalisation_ablation_scale_only" / f"output_{FREQUENCY}" / "comparison_with_standardscaler"
OUT_DIR.mkdir(parents=True, exist_ok=True)

FOLDERS = {"G7": "FX", "EM": "EM FX", "MSCI": "MSCI", "Rates": "Rates",
           "REIT": "REIT", "Equities": "Equities", "ETF": "ETF", "Crypto": "Crypto"}
SUFFIX = "_no_preprocess_lookback-250_horizon-20_slide-1"
ENDING = "_garch-t-1-1-simulated"
XCOLS = [f"X_{i}" for i in range(250)]
YCOLS = [f"Y_{i}" for i in range(20)]
LEVELS = np.arange(1, 10) / 10

def load_run(point_path, truth):
    points = pd.read_csv(point_path)
    # The forecast CSV is a copy of the input with Y columns replaced.
    # Verify the full row order using all untouched columns, including dates if present.
    pd.testing.assert_frame_equal(
        points.drop(columns=YCOLS).reset_index(drop=True),
        truth.drop(columns=YCOLS).reset_index(drop=True),
        check_dtype=False, check_exact=False, rtol=1e-12, atol=0,
    )
    with np.load(point_path.with_name(point_path.stem + "_quantiles.npz")) as saved:
        q = np.asarray(saved["quantiles"], dtype=float)
        levels = np.asarray(saved["quantile_levels"], dtype=float)
    if levels.ndim != 1 or q.shape != (len(truth), 20, len(levels)):
        raise ValueError(f"Unexpected quantile layout: {q.shape}; levels={levels.shape}")
    indices = []
    for tau in LEVELS:
        hits = np.flatnonzero(np.isclose(levels, tau, rtol=0, atol=1e-6))
        if len(hits) != 1:
            raise ValueError(f"Missing/duplicate quantile {tau}")
        indices.append(int(hits[0]))
    return points[YCOLS].to_numpy(float), q[:, :, indices]

def pinball_loss(y, q, tau):
    e = y - q
    return np.maximum(tau * e, (tau - 1) * e)

def score_run(x, y, p, q):
    rmse = np.sqrt(np.mean((y - p) ** 2, axis=0)).mean()
    qcrps = 2 * pinball_loss(y[:, :, None], q, LEVELS).mean()
    reference = np.quantile(x, .1, axis=1)[:, None]
    reference_loss = pinball_loss(y, reference, .1).mean()
    qss = (1 - pinball_loss(y, q[:, :, 0], .1).mean() / reference_loss
           if reference_loss > 0 else np.nan)
    return {"rmse": float(rmse), "qcrps": float(qcrps), "qss": float(qss)}

scale_paths = sorted(SCALE_ROOT.glob(f"*/test_*{SUFFIX}{ENDING}.csv"))
if not scale_paths:
    raise FileNotFoundError(f"No scale-only prediction CSVs under {SCALE_ROOT}")

paired_rows, error_rows = [], []
for scale_path in scale_paths:
    folder = scale_path.parent.name
    dataset = scale_path.stem.removeprefix("test_").removesuffix(SUFFIX + ENDING)
    try:
        truth = pd.read_csv(DATA_ROOT / folder / f"test_{dataset}{SUFFIX}.csv")
        x, y = truth[XCOLS].to_numpy(float), truth[YCOLS].to_numpy(float)
        ps, qs = load_run(scale_path, truth)
        pz, qz = load_run(STD_ROOT / folder / scale_path.name, truth)
        valid_s = np.isfinite(ps).all(axis=1) & np.isfinite(qs).all(axis=(1, 2))
        valid_z = np.isfinite(pz).all(axis=1) & np.isfinite(qz).all(axis=(1, 2))
        common = valid_s & valid_z & np.isfinite(x).all(axis=1) & np.isfinite(y).all(axis=1)
        if not common.any():
            raise ValueError("No common finite forecast windows")
        row = {"model": "Student-t GARCH", "asset_class": FOLDERS[folder], "dataset": dataset,
               "n_test_origins": len(y), "n_valid_scale_only": int(valid_s.sum()),
               "n_valid_scaler": int(valid_z.sum()), "n_common_valid": int(common.sum()),
               "n_excluded": int((~common).sum())}
        for label, p, q in [("scale_only", ps, qs), ("scaler", pz, qz)]:
            scores = score_run(x[common], y[common], p[common], q[common])
            row.update({f"{metric}_{label}": value for metric, value in scores.items()})
        paired_rows.append(row)
    except Exception as exc:
        error_rows.append({"asset_folder": folder, "dataset": dataset, "error": str(exc)})

scale_only_errors = pd.DataFrame(error_rows, columns=["asset_folder", "dataset", "error"])
scale_only_errors.to_csv(OUT_DIR / f"{FREQUENCY}_student_t_scale_only_comparison_errors.csv", index=False)
if not paired_rows:
    raise RuntimeError(f"No paired results. Check paths and the errors CSV in {OUT_DIR}.")

scale_only_comparison = pd.DataFrame(paired_rows)
summary_rows = []
for metric in ["rmse", "qcrps", "qss"]:
    a, b = f"{metric}_scale_only", f"{metric}_scaler"
    # Positive delta always means StandardScaler performed better.
    delta = (scale_only_comparison[b] - scale_only_comparison[a] if metric == "qss"
             else scale_only_comparison[a] - scale_only_comparison[b])
    scale_only_comparison[f"delta_{metric}"] = delta
    finite = np.isfinite(scale_only_comparison[[a, b]]).all(axis=1)
    tied = np.isclose(scale_only_comparison[a], scale_only_comparison[b], rtol=1e-6, atol=1e-12)
    n = int(finite.sum())
    summary_rows.append({
        "metric": metric, "n_paired_datasets": n,
        "mean_scale_only": scale_only_comparison.loc[finite, a].mean(),
        "mean_scaler": scale_only_comparison.loc[finite, b].mean(),
        "median_delta": delta[finite].median(),
        "scale_only_wins": int((finite & ~tied & (delta < 0)).sum()),
        "scaler_wins": int((finite & ~tied & (delta > 0)).sum()),
        "ties": int((finite & tied).sum()),
    })
    if metric != "qss":
        scale_only_comparison[f"{metric}_scale_only_over_scaler"] = (
            scale_only_comparison[a] / scale_only_comparison[b].replace(0, np.nan))

scale_only_summary = pd.DataFrame(summary_rows)
scale_only_comparison.to_csv(OUT_DIR / f"{FREQUENCY}_student_t_scale_only_vs_scaler_by_dataset.csv", index=False)
scale_only_summary.to_csv(OUT_DIR / f"{FREQUENCY}_student_t_scale_only_vs_scaler_summary.csv", index=False)
print(f"Scale-only files found: {len(scale_paths)}; paired datasets: {len(paired_rows)}; errors: {len(error_rows)}")
print("Subset comparison: only datasets with saved scale-only forecasts are included.")
print("Positive delta = StandardScaler better; RMSE/qCRPS ratio > 1 = StandardScaler better.")
print("Ties use rtol=1e-6, atol=1e-12. QSS uses the same rolling empirical q0.1 reference.")
print(scale_only_summary.to_string(index=False))
print("\nPaired results (worst scale-only qCRPS first):")
print(scale_only_comparison.sort_values("qcrps_scale_only_over_scaler", ascending=False).to_string(index=False))
if len(scale_only_errors):
    print("\nErrors / missing pairs:\n", scale_only_errors.to_string(index=False))
print("\nSaved to:", OUT_DIR)

# Full-ranking sensitivity

Compare within-dataset model rankings under the two preprocessing conditions.

Calculate:

1. Mean absolute rank change for each model,
2. Percentage of datasets where the best model remains unchanged,
3. Overall mean model ranks under each preprocessing condition,

separately for RMSE, CRPS, and QSS.

In [ ]:
import numpy as np
import pandas as pd

# ============================================================
# CONFIG
# ============================================================

METRICS = {
    "rmse": True,    # lower is better
    "qcrps": True,   # lower is better
    "qss": False,    # higher is better
}

EXPECTED_MODELS = {
    "rmse": 11,
    "qcrps": 11,
    "qss": 11,
}

KEYS = ["asset_class", "dataset"]

rank_rows = []
winner_rows = []

# ============================================================
# DATASET-LEVEL RANKS
# ============================================================

for metric, ascending in METRICS.items():

    scaler_col = f"{metric}_scaler"
    no_scaler_col = f"{metric}_no_scaler"

    sub = comparison.dropna(
        subset=[scaler_col, no_scaler_col]
    ).copy()

    for (asset_class, dataset), group in sub.groupby(KEYS):

        if len(group) != EXPECTED_MODELS[metric]:
            raise ValueError(
                f"{metric}: {asset_class}/{dataset} has "
                f"{len(group)} models, expected "
                f"{EXPECTED_MODELS[metric]}"
            )

        # Average ranks handle ties consistently
        std_ranks = group[scaler_col].rank(
            ascending=ascending,
            method="average"
        )

        raw_ranks = group[no_scaler_col].rank(
            ascending=ascending,
            method="average"
        )

        for idx in group.index:

            rank_rows.append({
                "metric": metric,
                "asset_class": asset_class,
                "dataset": dataset,
                "model": group.loc[idx, "model"],
                "rank_scaler": std_ranks.loc[idx],
                "rank_no_scaler": raw_ranks.loc[idx],
                "abs_rank_change": abs(
                    std_ranks.loc[idx] - raw_ranks.loc[idx]
                ),
            })

        # Check whether the best model changes
        # Treat tied winners as sets.
        if ascending:
            best_std = group[scaler_col].min()
            best_raw = group[no_scaler_col].min()
        else:
            best_std = group[scaler_col].max()
            best_raw = group[no_scaler_col].max()

        std_winners = set(
            group.loc[
                group[scaler_col] == best_std,
                "model"
            ]
        )

        raw_winners = set(
            group.loc[
                group[no_scaler_col] == best_raw,
                "model"
            ]
        )

        winner_rows.append({
            "metric": metric,
            "asset_class": asset_class,
            "dataset": dataset,
            "same_winner": std_winners == raw_winners,
            "winner_scaler": "; ".join(sorted(std_winners)),
            "winner_no_scaler": "; ".join(sorted(raw_winners)),
        })


# ============================================================
# SUMMARIES
# ============================================================

rank_df = pd.DataFrame(rank_rows)
winner_df = pd.DataFrame(winner_rows)

model_rank_summary = (
    rank_df.groupby(["metric", "model"], as_index=False)
    .agg(
        mean_rank_scaler=("rank_scaler", "mean"),
        mean_rank_no_scaler=("rank_no_scaler", "mean"),
        mean_abs_rank_change=("abs_rank_change", "mean"),
        pct_rank_unchanged=(
            "abs_rank_change",
            lambda x: 100 * np.mean(x == 0),
        ),
    )
)

winner_summary = (
    winner_df.groupby("metric", as_index=False)
    .agg(
        n_datasets=("dataset", "count"),
        pct_same_winner=(
            "same_winner",
            lambda x: 100 * np.mean(x),
        ),
    )
)

# ============================================================
# PRINT
# ============================================================

print("\n" + "=" * 80)
print("BEST-MODEL STABILITY")
print("=" * 80)

print(
    winner_summary.round(3).to_string(index=False)
)

for metric in METRICS:

    print("\n" + "=" * 80)
    print(f"{metric.upper()} MODEL RANKINGS")
    print("=" * 80)

    temp = model_rank_summary[
        model_rank_summary["metric"] == metric
    ].sort_values("mean_rank_scaler")

    print(
        temp.round(3).to_string(index=False)
    )

# ============================================================
# SAVE
# ============================================================

rank_df.to_csv(
    OUT_DIR / "daily_normalisation_full_rank_changes_with_timestral.csv",
    index=False,
)

winner_df.to_csv(
    OUT_DIR / "daily_normalisation_winner_changes_with_timestral.csv",
    index=False,
)

model_rank_summary.to_csv(
    OUT_DIR / "daily_normalisation_full_rank_summary_with_timestral.csv",
    index=False,
)

winner_summary.to_csv(
    OUT_DIR / "daily_normalisation_winner_summary_with_timestral.csv",
    index=False,
)